In [1]:
### LIBRARIES

# Data manipulation
import pandas as pd
import numpy as np

# Financial data retrieval
import yfinance as yf

# Data visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# File management
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Project directory
PROJECT_DIR = Path.cwd().parent

# Data directories
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Create directories if they do not exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Historical sample
START_DATE = "2021-10-12"
END_DATE = "2026-10-10"

# Financial instruments
TICKERS = ["SPY", "QQQ", "GLD", "TLT", "MSFT"]

# Exchange rate
FX_TICKER = "GBPUSD=X"

# Portfolio specifications
INITIAL_CAPITAL_GBP = 100000

print("Project parameters established.")


Project parameters established.


In [4]:
# Download historical asset prices
asset_data = yf.download(
    tickers=TICKERS,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=True,
    progress=False
)

# Extract adjusted closing prices
asset_prices_usd = asset_data["Close"]

# Inspect dataset
print(asset_prices_usd.head())
print(asset_prices_usd.tail())

print("Dataset dimensions:", asset_prices_usd.shape)

print("\nNumber of observations:", asset_prices_usd.shape[0])

print("\nNumber of instruments:", asset_prices_usd.shape[1])

print("\nInstrument names:")
print(asset_prices_usd.columns.tolist())

asset_prices_usd.to_csv(
    RAW_DIR / "asset_prices_usd.csv"
)

print("USD asset prices saved successfully.")

# Download historical GBP/USD exchange rates
fx_data = yf.download(
    tickers=FX_TICKER,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=True,
    progress=False
)

# Extract closing exchange rate
gbpusd = fx_data["Close"]

# Ensure the series is one-dimensional
if isinstance(gbpusd, pd.DataFrame):
    gbpusd = gbpusd.iloc[:, 0]

gbpusd = gbpusd.rename("GBPUSD")

print(gbpusd.head())
print(gbpusd.tail())
print("Exchange rate series dimensions:", gbpusd.shape)

gbpusd.to_csv(
    RAW_DIR / "gbpusd_exchange_rate.csv"
)
print("GBP/USD exchange rate data saved successfully.")

Ticker          GLD     MSFT      QQQ      SPY      TLT
Date                                                   
2021-10-12 164.6600 281.0849 346.6237 405.5659 120.3343
2021-10-13 167.5900 284.3768 349.3994 407.0248 121.5048
2021-10-14 168.0000 290.5575 355.8342 413.8713 121.9562
2021-10-15 165.3300 291.9586 358.0762 417.0233 121.2456
2021-10-18 164.9200 294.9146 361.6769 418.2580 121.8141
Ticker          GLD     MSFT      QQQ      SPY     TLT
Date                                                  
2026-10-05 379.5500 525.1800 756.2000 774.8300 77.1100
2026-10-06 382.2700 529.3000 759.6600 779.0900 77.2800
2026-10-07 375.8800 529.7600 757.7300 777.2200 77.1500
2026-10-08 378.6200 522.6100 747.5800 773.9300 77.8700
2026-10-09 384.5800 535.0700 751.2700 778.5700 77.9800
Dataset dimensions: (1254, 5)

Number of observations: 1254

Number of instruments: 5

Instrument names:
['GLD', 'MSFT', 'QQQ', 'SPY', 'TLT']
USD asset prices saved successfully.
Date
2021-10-12   1.3591
2021-10-13   1.3579

In [5]:
### final inspection and verification
print("ASSET DATA INFORMATION")

print("\nDataset information:")
print(asset_prices_usd.info())

print("\nDescriptive statistics:")
print(asset_prices_usd.describe())

print("\nMissing observations:")
print(asset_prices_usd.isna().sum())

print("\nDuplicate dates:")
print(asset_prices_usd.index.duplicated().sum())

print("EXCHANGE RATE DATA INFORMATION")

print("\nDataset information:")
print(gbpusd.info())

print("\nDescriptive statistics:")
print(gbpusd.describe())

print("\nMissing observations:")
print(gbpusd.isna().sum())

print("\nDuplicate dates:")
print(gbpusd.index.duplicated().sum())

print("ASSET PRICE DATE RANGE")

print("Start:", asset_prices_usd.index.min())
print("End:", asset_prices_usd.index.max())

print("\nEXCHANGE RATE DATE RANGE")

print("Start:", gbpusd.index.min())
print("End:", gbpusd.index.max())

ASSET DATA INFORMATION

Dataset information:
<class 'pandas.DataFrame'>
DatetimeIndex: 1254 entries, 2021-10-12 to 2026-10-09
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   GLD     1254 non-null   float64
 1   MSFT    1254 non-null   float64
 2   QQQ     1254 non-null   float64
 3   SPY     1254 non-null   float64
 4   TLT     1254 non-null   float64
dtypes: float64(5)
memory usage: 58.8 KB
None

Descriptive statistics:
Ticker       GLD      MSFT       QQQ       SPY       TLT
count  1254.0000 1254.0000 1254.0000 1254.0000 1254.0000
mean    249.3931  368.7432  450.5428  517.9904   89.1601
std      92.8826   85.6886  135.1769  124.3771   11.3139
min     151.2300  207.3437  254.3989  338.5378   72.8334
25%     176.9150  294.9434  341.5809  409.7398   82.7306
50%     212.9650  384.0170  432.7059  498.4262   85.0815
75%     309.0300  423.1789  552.9753  615.8105   90.4489
max     495.9000  537.6464  759.6600  779.0900  129.3586

# There are 45 more FX observations than asset-price observations.
# Data Alignment required

In [ ]:
### VISUALIZE RAW DATA
## Asset Prices
plt.figure(figsize=(14, 7))

for ticker in TICKERS:
    plt.plot(
        asset_prices_usd.index,
        asset_prices_usd[ticker],
        label=ticker
    )

plt.title("Historical Asset Prices (USD)")
plt.xlabel("Date")
plt.ylabel("Adjusted Closing Price (USD)")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

## GBP/USD Exchange Rate
plt.figure(figsize=(14, 5))

plt.plot(
    gbpusd.index,
    gbpusd,
    label="GBP/USD"
)

plt.title("Historical GBP/USD Exchange Rate")
plt.xlabel("Date")
plt.ylabel("USD per GBP")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()